In [1]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')


import openai
import os

# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
import json
with open(f"{ROOT_DIR}/API_KEYS2.json", "r") as file:
    api_keys = json.load(file)

os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
os.environ['CACHE_DIR'] = os.path.join(ROOT_DIR, 'cache_dir3')

# Mass Maps

In [2]:
import torch
from datasets import load_dataset

test_dataset = load_dataset("BrachioLab/massmaps-cosmogrid-100k", split='test')
test_dataset.set_format('torch', columns=['input', 'label'])

In [3]:
# import importlib
import sys; sys.path.append("../src")
# import massmaps
# importlib.reload(massmaps)
from massmaps import MassMapsExample
from massmaps import massmap_to_pil_norm, get_llm_generated_answer, get_llm_output
from massmaps import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores
from llms import load_model

In [4]:
from tqdm.auto import tqdm
import json

In [5]:
# model = 'gpt-4o'
models = [
    'gpt-4o',
    'claude-3-5-sonnet-latest',
    'gemini-2.0-flash',
    'o1'
]

eval_model_name = 'gemini-2.0-flash'
eval_model = load_model(eval_model_name)



In [6]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [7]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}_{eval_model_name}.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            example_dict = result
            
            example = MassMapsExample(
                input = torch.tensor(example_dict['input']).to(device),
                answer = example_dict['answer'],
                llm_answer = example_dict['llm_answer'],
                llm_explanation = example_dict['llm_explanation'],
            )
            
            # isolate individual features
            claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if claims is None:
                continue
            example.claims = [claim.strip() for claim in claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example.input, 
                example.llm_answer,
                example.claims,
                model=eval_model
            )
            example.relevant_claims = relevant_claims

            # calculate expert alignment scores
            align_infos = calculate_expert_alignment_scores(
                example.relevant_claims, 
                eval_model,
            )

            alignable_claims = [info["Claim"] for info in align_infos]
            alignment_categories = [info["Category"] for info in align_infos]
            aligned_category_ids = [info["Category ID"] for info in align_infos]
            alignment_scores = [info["Alignment"] for info in align_infos]
            alignment_raws = [info["Alignment Raw"] for info in align_infos]
            alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
            example.alignable_claims = alignable_claims
            example.alignment_categories = alignment_categories
            example.aligned_category_ids = aligned_category_ids
            example.alignment_scores = alignment_scores
            example.alignment_raws = alignment_raws
            example.alignment_reasonings = alignment_reasonings
            
            # save
            save_dict = {}
            for k, v in example.__dict__.items():
                save_dict[k] = v if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
            # with open(save_path, 'wt') as output_file:
            #     json.dump(save_dict, output_file)

            new_results.append(save_dict)


        with open(save_path, 'wt') as output_file:
            json.dump(new_results, output_file, indent=4)

=== Using model gpt-4o ===
=== Using method vanilla ===


  0%|          | 0/100 [00:00<?, ?it/s]

=== Using model claude-3-5-sonnet-latest ===
=== Using method vanilla ===


  0%|          | 0/100 [00:00<?, ?it/s]

Error calling Google's API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Error calling Google's API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Error calling Google's API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Error calling Google's API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Error calling Google's API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Error calling Google's API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Error calling Google's API: 503 UNAVAILABLE. {

  0%|          | 0/100 [00:00<?, ?it/s]

=== Using model o1 ===
=== Using method vanilla ===


  0%|          | 0/100 [00:00<?, ?it/s]

# Cholec

# Emotion